In [1]:
import pandas as pd
import os
import sys
import numpy as np

root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root not in sys.path:
    sys.path.insert(0, root)

from GestorCache import gestorCache

carpeta = "data"
ruta_archivo = os.path.join(carpeta, "dataset_PostNLP.csv")

df_sucio = []
df_sucio = pd.read_csv(ruta_archivo)

# Imprimimos el head
print("-" * 30)
print(df_sucio.dtypes)

------------------------------
id                        object
site                      object
job_url                   object
job_url_direct            object
title                     object
company                   object
location                  object
date_posted               object
job_type                  object
salary_source             object
interval                  object
min_amount               float64
max_amount               float64
currency                  object
is_remote                 object
job_level                 object
job_function              object
listing_type              object
emails                    object
description               object
company_industry          object
company_url               object
company_logo              object
company_url_direct        object
company_addresses         object
company_num_employees     object
company_revenue           object
company_description       object
skills                   float64
experience_r

In [2]:
df_clean = df_sucio.copy()
df_clean.columns

Index(['id', 'site', 'job_url', 'job_url_direct', 'title', 'company',
       'location', 'date_posted', 'job_type', 'salary_source', 'interval',
       'min_amount', 'max_amount', 'currency', 'is_remote', 'job_level',
       'job_function', 'listing_type', 'emails', 'description',
       'company_industry', 'company_url', 'company_logo', 'company_url_direct',
       'company_addresses', 'company_num_employees', 'company_revenue',
       'company_description', 'skills', 'experience_range', 'company_rating',
       'company_reviews_count', 'vacancy_count', 'work_from_home_type',
       'search_location', 'search_query'],
      dtype='object')

In [3]:
#Funciones auxiliares
def verRecuentoTipos(df):
    for col in df.columns:
        print(f"Columna: {col}")
        print(df[col].value_counts())
        print("-" * 30)

In [ ]:
def tratamiento(df):

    # Defino las funciones aqui por encapsulamiento, aunque podrían estar fuera
    # Indico donde empiezan las funciones para que sea más fácil de leer
    def dropearColumnas(df):
        # Lista de columnas a eliminar
        cols_to_drop = ['id', 'job_url', 'job_url_direct', 'title', 'emails',
                        'company_url', 'company_logo', 'company_url_direct',
                        'company_description','search_location'
                        ]

        # Usamos errors='ignore' para que no rompa el código si la columna ya fue borrada
        df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

        return df
    
    def arreglandoTipos(df):
        mapeoTipos = {
        'company' : 'string',
        'location' : 'string',
        'job_type' : 'string',
        'job_level' : 'string',
        'job_function' : 'string',
        'company_industry' : 'string'
        }
        df = df.astype(mapeoTipos)

        df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')

        return df

    def obtenerTodosLosNan(df):

        # Para los valores de texto
        valores_a_limpiar = [ 
        '[]', 
        '', 
        'nan', 
        'None', 
        'Unknown', 
        'N/A'
        '<NA>'
        ]

        df = df.replace(valores_a_limpiar, np.nan)

        # para los valores numéricos, si el valor es 0 lo consideramos como NaN
        cols_num = ['min_amount', 'max_amount']
        for col in cols_num:
            df[col] = df[col].apply(lambda x: np.nan if x == 0 else x)

        # Opcional: Limpiar automáticamente columnas que estén totalmente vacías (NaN)
        temp = df.dropna(axis=1, how='all')

        print(f"Columnas eliminadas por estar completamente vacías: {set(df.columns) - set(temp.columns)}")
        
        PORCENTAJE_DE_NAN_ACEPTADOS = 0.2

        umbral_minimo = len(df) * PORCENTAJE_DE_NAN_ACEPTADOS

        # Borramos columnas (axis=1) que no lleguen a ese mínimo de datos reales
        df = temp.dropna(thresh=umbral_minimo, axis=1)

        print(f"Columnas eliminadas por tener más del {PORCENTAJE_DE_NAN_ACEPTADOS*100}% de NaN: {set(temp.columns) - set(df.columns)}")

        return df

    def discretizar(df):

        # Calificamos si se encuentran en el top 5%, en el 5% de los peores o en el intermedio de numero de ofertas encontradas
        def companiasSegunVacantes(col):

            def empresasEnPercentil(col, tail, percent = 0.975):
                numeroAps = col.value_counts()
                
                if tail:
                    percent = 1 - percent

                umbral = numeroAps.quantile(percent).astype(int)

                if tail:
                    seleccionados = numeroAps[numeroAps <= umbral]
                else:
                    seleccionados = numeroAps[numeroAps >= umbral]

                return seleccionados.index

            # ------- Fin declaracion de funciones -------
            
            top_empresas = empresasEnPercentil(df['company'], tail=False)
            tail_empresas = empresasEnPercentil(df['company'], tail=True)

            out = pd.Series(index=col.unique(), dtype=int)

            #Si la empresa está en el top, le asignamos 2, si está en el tail le asignamos 0, y si no, le asignamos 1
            for empresa in col:
                if empresa in top_empresas: #Top empresas
                    out[empresa] = 2
                elif empresa in tail_empresas: #Tail empresas
                    out[empresa] = 0
                else: # Empresas intermedias
                    out[empresa] = 1
            
            return out

        # Obtenemos [Pais, Ciudad] de cada localización, si precision es False se devuelve solo el pais
        def localizacion(col, precision=True):

            cache = gestorCache.getGeoCache()
            
            valores_unicos = col.unique()
            misses = [v for v in valores_unicos if v not in cache and pd.notna(v)]
            
            if misses:
                cache = gestorCache.actualizar_misses(misses)

            localizacion_normalizada = col.map(cache)
            
            if precision:
                return localizacion_normalizada

            return localizacion_normalizada.str[0]



        # -----------------------------------------------------------
        # ------------- FIN DECLARACION DE FUNCIONES ----------------
        # -----------------------------------------------------------


        df['site'] = pd.factorize(df['site'])[0]

        df['companySegunVacantes'] = companiasSegunVacantes(df['company'])
        df.drop(columns=['company'], inplace=True)

        df['location'] = localizacion(df['location'])
        
        #verRecuentoTipos(df)

        return df
        
    # -------------------------------------------------------
    # ------------ FIN DECLARACION DE FUNCIONES -------------
    # -------------------------------------------------------

    df = dropearColumnas(df)

    df = arreglandoTipos(df)

    df = obtenerTodosLosNan(df)

    df = discretizar(df)

    return df

In [5]:
df_clean = tratamiento(df_clean)
print(df_clean['location'])

Columnas eliminadas por estar completamente vacías: {'work_from_home_type', 'experience_range', 'company_reviews_count', 'company_rating', 'vacancy_count', 'skills'}
Columnas eliminadas por tener más del 20.0% de NaN: {'company_num_employees', 'company_revenue', 'company_addresses'}
0                  NaN
1               España
2               España
3               España
4               España
             ...      
10542              NaN
10543              NaN
10544    United States
10545    United States
10546    United States
Name: location, Length: 10547, dtype: object


In [6]:
ruta_salida = os.path.join(carpeta, "dataset_entrenamiento.csv")
df_clean.to_csv(ruta_salida, index=False)